# 第3次课：字符串处理与输入输出

**地学编程基础** | 2学时（90分钟）

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **掌握**字符串的索引（`s[n]`）和切片（`s[a:b]`）操作；
2. **熟练运用**字符串的5大常用方法：`split`、`strip`、`replace`、`join`、`find`；
3. **掌握** `input()` 和 `print()` 的进阶用法（提示语、sep/end 参数、转义字符）；
4. **能够**解析常见的地学字符串数据（度分秒坐标、GPS NMEA 语句、地名列表等）；
5. **具备**清洗"脏数据"的初步能力——这是地学数据处理的日常工作。

---

## 🔄 上次课回顾 + 一道思考题

上次课最后，留了一道思考题：

> 字符串 `s = "3.14abc"`，怎么提取出 `3.14`？

今天就是要解决这个问题——并且解决比这个**复杂得多**的真实地学场景。

**你将学会**：
- 把 `"39°54'27\"N"` 这样的度分秒字符串解析成 `39.9075` 度
- 把 `"  杭州市  \n"`这样带各种空白字符的脏地名清洗干净
- 从 `$GPGGA,123519,4807.038,N,01131.000,E,...` 这样的 GPS 语句里提取经纬度

**这些都是地学数据处理中天天会遇到的真实场景。**

---

## 一、字符串索引与切片

上次课我们提过字符串的索引（`s[0]`），今天系统讲一下。

### 1.1 字符串的索引：每个字符的位置编号

**Python 中字符串里每个字符都有一个位置编号——从 0 开始，从左往右数。**

In [ ]:
s = "Hello GIS"
#    0 1 2 3 4 5 6 7 8
#    H e l l o   G I S

print(s[0])    # 第0个字符：H
print(s[6])    # 第6个字符：G
print(s[8])    # 第8个字符（最后一个）：S

# 试试越界会怎么样？
# print(s[100])    # IndexError: string index out of range

### 1.2 负索引：从右边数

**Python 还允许用负数索引——从右边往左数，最后一个字符是 -1。**

In [ ]:
s = "Hello GIS"
#     0 1 2 3 4 5 6 7 8     ← 正索引
#    -9-8-7-6-5-4-3-2-1     ← 负索引

print(s[-1])    # 倒数第1个：S
print(s[-2])    # 倒数第2个：I
print(s[-9])    # 倒数第9个（即正数第0个）：H

**为什么要有负索引？** 因为我们经常需要取字符串的**最后几个字符**：

- 取文件名最后4个字符判断扩展名（`.tif`、`.shp`）
- 取坐标字符串最后1个字符判断方向（`N`/`S`/`E`/`W`）

In [ ]:
# 实际地学例子：判断坐标方向
lat_str = "39.9075N"
direction = lat_str[-1]    # 取最后一个字符
print(f"方向：{direction}")    # N

# 文件扩展名
filename = "landsat_clip.tif"
ext = filename[-4:]         # 注意这里用了切片，下面就讲
print(f"扩展名：{ext}")

### 1.3 字符串切片：取一段而不是一个字符

**切片（slicing）**：用 `s[a:b]` 取**第 a 到 第 b（不含）** 之间的子串。

**🔑 切片三要素：起始[a]、结束[b]、步长[c]**

- `s[a:b]` —— 从 a 到 b（不含 b）
- `s[a:b:c]` —— 从 a 到 b（不含 b），步长为 c
- `s[a:]` —— 从 a 到末尾
- `s[:b]` —— 从开头到 b（不含 b）
- `s[:]` —— 完整复制

In [ ]:
s = "Hello GIS"
#    0 1 2 3 4 5 6 7 8

print(s[0:5])     # 从0到5（不含5）：Hello
print(s[6:9])     # 从6到9（不含9）：GIS
print(s[6:])      # 从6到末尾：GIS
print(s[:5])      # 从开头到5：Hello
print(s[:])       # 完整复制：Hello GIS

**⚠️ 切片的'左闭右开'特性**：

`s[0:5]` 取的是位置 **0、1、2、3、4** 共5个字符，**不包括位置5**。

这种'左闭右开'是 Python 的统一规则，将来 NumPy、Pandas 切片都是这样。**先记住这个规律，将来一通百通。**

In [ ]:
# 切片也支持负索引
s = "landsat_clip.tif"

print(s[-4:])     # 最后4个字符：.tif
print(s[:-4])     # 除了最后4个：landsat_clip
print(s[-3:])     # 最后3个字符：tif

In [ ]:
# 步长（了解即可）
s = "ABCDEFGHIJ"

print(s[::2])     # 步长2，每隔1个取1个：ACEGI
print(s[::-1])    # 步长-1，反向：JIHGFEDCBA（这是反转字符串的常见技巧！）

**💡 `s[::-1]` 是反转字符串的经典写法**——值得记一下，将来用得到。

---

## 🎯 即时练习①

**练习目标**：熟练使用字符串索引与切片。

**练习题**：给定字符串 `s = "NorthLatitude39.9075E116.4074"`（这是某种简化的GPS数据格式）：

1. 用索引取出第一个字符（应该是 `N`）；
2. 用切片取出 `"NorthLatitude"` 这部分（前13个字符）；
3. 用切片取出 `"39.9075"` 这部分（位置13到20）；
4. 用切片取出 `"116.4074"` 这部分（位置21到29）；
5. 用负索引取出最后一个字符；
6. 反转整个字符串。

**提示**：你可以先用 `print(s[i])` 把几个关键位置的字符打出来，验证位置编号是否正确。

In [ ]:
# ===== 即时练习① =====
# 在下方编写你的代码：

s = "NorthLatitude39.9075E116.4074"





---

## 二、字符串的5大常用方法

字符串对象自带很多有用的'方法'（你可以理解为字符串自带的'技能'）。今天我们重点掌握**5个最常用的**：

| 方法 | 作用 | 用途 |
|---|---|---|
| `split` | 分割字符串 | 解析CSV、解析坐标 |
| `strip` | 去除两端空白 | 清洗用户输入、清洗读到的数据 |
| `replace` | 替换内容 | 去掉单位、统一格式 |
| `join` | 拼接字符串 | 把多个字符串连起来 |
| `find` | 查找子串位置 | 判断是否包含、定位关键字 |

**🔑 调用方法的语法**：`字符串.方法名(参数)`，例如 `"hello".upper()` 

### 2.1 split —— 分割字符串

`split(分隔符)` 按指定字符把字符串**切成多块**，返回一个列表（list，下次课详细学）。

In [ ]:
# 例1：按逗号分割（典型 CSV 数据）
line = "杭州,30.27,120.16,41.7"
parts = line.split(",")
print(parts)    # ['杭州', '30.27', '120.16', '41.7']

# 用索引取出各部分
print(parts[0])    # 杭州
print(parts[1])    # 30.27
print(parts[2])    # 120.16

In [ ]:
# 例2：按空格分割
address = "浙江省 杭州市 西湖区"
parts = address.split(" ")
print(parts)    # ['浙江省', '杭州市', '西湖区']

In [ ]:
# 例3：不传参数 —— 默认按所有空白字符（空格、Tab、换行）分割
messy = "杭州   北京\t上海\n广州"
cities = messy.split()    # 不传参数
print(cities)    # ['杭州', '北京', '上海', '广州']

**💡 split 不传参数是个非常实用的技巧**——它能处理混乱的空白字符（多个空格、Tab、换行），把脏数据清洗成整齐的列表。

### 2.2 strip —— 去除两端空白

`strip()` 去除字符串**两端**的空白字符（不影响中间）。

**这是处理用户输入和文件读取数据的'必备清洁工具'**——因为这些数据经常带有看不见的空格、换行符。

In [ ]:
# 经典场景：去除前后空格
messy = "   杭州市  "
clean = messy.strip()
print(f"|{messy}|")    # |   杭州市  |
print(f"|{clean}|")    # |杭州市|

In [ ]:
# 去除换行符（从文件读取数据时常见）
line_from_file = "39.9075\n"      # \n 就是换行符
clean = line_from_file.strip()
print(f"|{line_from_file}|")
print(f"|{clean}|")

In [ ]:
# strip 还能去除指定的字符
s = "###杭州市###"
print(s.strip("#"))    # 杭州市

# 还可以去除多种字符（任何出现在两端的指定字符都会被去掉）
s = "<<杭州市>>"
print(s.strip("<>"))   # 杭州市

**📌 strip 的兄弟方法**：
- `lstrip()`：只去除**左侧**空白
- `rstrip()`：只去除**右侧**空白

### 2.3 replace —— 替换内容

`replace(旧内容, 新内容)` 把字符串里所有的**旧内容替换成新内容**。

**地学数据清洗的利器**——经常需要去掉多余的单位、符号、统一格式。

In [ ]:
# 经典场景：去掉数字后面的单位（解决上节课的思考题！）
elevation_str = "8848.86米"
elevation_num = elevation_str.replace("米", "")    # 把'米'替换成空字符串，相当于删除
print(elevation_num)              # 8848.86
print(float(elevation_num))       # 现在能转成 float 了！

In [ ]:
# 例：把空格全部替换成下划线（生成文件名常用）
title = "Lesson 03 String Processing"
filename = title.replace(" ", "_")
print(filename)    # Lesson_03_String_Processing

In [ ]:
# 例：连续替换（链式调用）
messy = "39°54'27\"N"
# 一次去掉一种符号
cleaned = messy.replace("°", " ").replace("'", " ").replace("\"", " ")
print(cleaned)    # 39 54 27 N

### 2.4 join —— 拼接字符串

`join` 是 `split` 的反向操作——**把多个字符串拼接成一个**。

**🔑 语法有点反直觉**：`分隔符.join(字符串列表)`

In [ ]:
# 例1：用逗号拼接（生成 CSV 行）
parts = ["杭州", "30.27", "120.16", "41.7"]
line = ",".join(parts)
print(line)    # 杭州,30.27,120.16,41.7

In [ ]:
# 例2：用其他分隔符
cities = ["杭州", "北京", "上海", "广州"]

print(" → ".join(cities))    # 杭州 → 北京 → 上海 → 广州
print("\n".join(cities))     # 每个城市一行

### 2.5 find —— 查找子串位置

`find(子串)` 返回**子串第一次出现的位置编号**；如果找不到，返回 -1。

In [ ]:
s = "latitude=39.90; longitude=116.40"

print(s.find("="))         # 8（第一个=的位置）
print(s.find("longitude")) # 16
print(s.find("hello"))     # -1（找不到）

In [ ]:
# 实用例子：判断字符串是否包含某内容
filename = "landsat_band4.tif"

if filename.find(".tif") != -1:    # find 返回不是 -1 表示找到了
    print("这是一个 GeoTIFF 文件")

# 上面这种写法以后我们会用更优雅的 'in' 运算符代替（下次课讲）：
# if ".tif" in filename:

### 2.6 其他常用方法（了解即可）

下面这些用得没那么频繁，但偶尔会遇到：

In [ ]:
s = "Hello GIS"

print(s.upper())          # 全大写：HELLO GIS
print(s.lower())          # 全小写：hello gis
print(s.startswith("He")) # 是否以...开头：True
print(s.endswith("GIS"))  # 是否以...结尾：True
print(s.count("l"))       # 子串出现次数：2
print("abc123".isdigit()) # 是否全是数字：False
print("123".isdigit())    # 是否全是数字：True

---

## 🎯 即时练习②

**练习目标**：综合运用字符串方法清洗数据。

**练习题**：以下是从某文件读出来的脏地名数据：

```python
messy_cities = "  杭州市;;北京市；上海市 ；  广州市  "
```

请编写代码完成：

1. 把所有的中文分号 `；` 替换成英文分号 `;`（地学数据中混用半角全角是常见问题）；
2. 用 `;` 分割成城市列表；
3. 用 `strip()` 把每个城市名的两端空白去掉（注意：你可以**先 split 再分别处理**，也可以**先整体 strip 后再 split**）；
4. 把清洗好的城市名用 ` → ` 重新拼接成一个字符串。

**期望输出**：
```
杭州市 → 北京市 → 上海市 → 广州市
```

**提示**：
- 替换中文分号：`messy_cities.replace("；", ";")`
- 你可能需要用一个简单循环（下次课才学）来处理列表中的每个元素，**也可以暂时手动一个一个 strip**——本练习两种方式都接受。

In [ ]:
# ===== 即时练习② =====
# 在下方编写你的代码：

messy_cities = "  杭州市;;北京市；上海市 ；  广州市  "





---

## 三、input 和 print 进阶

前两节课我们已经用过 `input()` 和 `print()` 了。今天再补充几个**进阶用法**。

### 3.1 input() 加提示语 + 类型转换

上次课我们已经强调过：**`input()` 永远返回字符串**。今天再写一个完整的'地学小工具'例子：

In [ ]:
# 一个简单的'摄氏度转华氏度'小工具
# 注意：input 接收到的永远是字符串，必须先用 float() 转换

celsius_str = input("请输入摄氏温度：")
celsius = float(celsius_str)
fahrenheit = celsius * 9 / 5 + 32

print(f"摄氏 {celsius}°C = 华氏 {fahrenheit:.2f}°F")

### 3.2 print 的 sep 参数：自定义分隔符

`print()` 可以一次打印多个值，默认用**空格**分隔。但你可以用 `sep` 参数自定义：

In [ ]:
print("杭州", "北京", "上海")             # 默认空格分隔
print("杭州", "北京", "上海", sep=",")    # 用逗号分隔
print("杭州", "北京", "上海", sep=" → ")  # 用箭头分隔
print("杭州", "北京", "上海", sep="\n")   # 用换行分隔——每个一行

### 3.3 print 的 end 参数：控制结尾

`print()` 默认在打印后会**换行**（结尾加 `\n`）。可以用 `end` 参数改变这个行为：

In [ ]:
# 默认：每个 print 后换行
print("第一行")
print("第二行")

# 用 end="" 不换行（连成一行）
print("加载中", end="")
print("...")

### 3.4 转义字符：字符串中的'特殊符号'

上面我们看到了 `\n` 表示换行。这种用 `\` 开头的特殊符号叫**转义字符**。

| 转义字符 | 含义 |
|---|---|
| `\n` | 换行 |
| `\t` | 制表符（Tab） |
| `\\` | 反斜杠本身 |
| `\"` | 双引号 |
| `\'` | 单引号 |

In [ ]:
# 换行
print("第一行\n第二行")

print("---")

# 制表符（用于对齐）
print("杭州\t30.27\t120.16")
print("北京\t39.90\t116.40")
print("上海\t31.23\t121.47")

print("---")

# 字符串里要包含双引号
print("他说：\"GIS 很有意思\"")

**💡 文件路径里的反斜杠问题**

Windows 的文件路径用反斜杠（`C:\Users\data.tif`），但 `\U` 在 Python 里是另一种转义字符。**两种解决方法**：

In [ ]:
# 方法1：用双反斜杠转义
path1 = "C:\\Users\\data.tif"
print(path1)

# 方法2：用原始字符串（在引号前加 r）—— 推荐
path2 = r"C:\Users\data.tif"
print(path2)

# 方法3：用正斜杠（推荐，跨平台）
path3 = "C:/Users/data.tif"
print(path3)

---

## 四、综合实战：度分秒坐标解析

**这是地学领域的经典任务**——把人类可读的'度°分′秒″'坐标转换成计算机可用的十进制度。

### 4.1 度分秒 → 十进制度的公式

**例**：北纬 39°54'27"

**转换公式**：
$$
十进制度 = 度 + \frac{分}{60} + \frac{秒}{3600}
$$

$$
39 + \frac{54}{60} + \frac{27}{3600} = 39.9075°
$$

In [ ]:
# 先验证一下公式
degrees = 39
minutes = 54
seconds = 27

decimal = degrees + minutes / 60 + seconds / 3600
print(f"39°54'27\" = {decimal:.4f}°")

### 4.2 解析思路：从字符串到数字

现在的挑战是：度、分、秒都**埋藏在字符串 `"39°54'27\""`** 里。

**解析思路**：
1. 用 `replace` 把 `°`、`'`、`"` 这些符号都换成空格；
2. 用 `split` 按空格分割成三块；
3. 把每一块转成整数（`int()`）；
4. 套用公式计算。

**让我们一步步实现**：

In [ ]:
# 原始字符串（注意 " 在字符串里要用 \" 转义）
coord_str = "39°54'27\""
print(f"原始字符串：{coord_str}")

# Step 1: 把所有特殊符号替换成空格
cleaned = coord_str.replace("°", " ").replace("'", " ").replace("\"", " ")
print(f"清洗后：'{cleaned}'")

In [ ]:
# Step 2: 用 split() 不加参数 —— 自动按空白分割并去除空字符串
parts = cleaned.split()
print(f"分割后：{parts}")    # ['39', '54', '27']

In [ ]:
# Step 3: 把每部分转成整数
degrees = int(parts[0])
minutes = int(parts[1])
seconds = int(parts[2])

print(f"度：{degrees}, 分：{minutes}, 秒：{seconds}")

In [ ]:
# Step 4: 套用公式
decimal = degrees + minutes / 60 + seconds / 3600
print(f"{coord_str} = {decimal:.4f}°")

### 4.3 完整版：一个字符串到结果

把上面4步合在一起：

In [ ]:
coord_str = "39°54'27\""

# 一次性完成（链式调用 replace）
parts = coord_str.replace("°", " ").replace("'", " ").replace("\"", " ").split()

degrees = int(parts[0])
minutes = int(parts[1])
seconds = int(parts[2])

decimal = degrees + minutes / 60 + seconds / 3600

print(f"{coord_str} = {decimal:.4f}°")

**🎉 你刚才做的事情**：

把一个'度°分′秒″'格式的字符串，自动转换成了十进制度数。

**这就是真实的地学数据处理工作！** 你已经从'看着代码'升级到'能用代码处理真实问题'了。

**思考一下**：上面的代码还有什么不足？
- 如果是 `"39度54分27秒"` 这种中文格式，能处理吗？
- 如果是 `"39°54'27.5\""` 带小数的秒，能处理吗？
- 如果带方向标识 `"39°54'27\"N"`，怎么处理？

（这些都是接下来你可以挑战的方向。）

---

## 🎯 即时练习③（综合挑战）—— 解析GPS NMEA语句

**练习目标**：综合运用所学，解析 GPS 设备的真实输出。

**背景知识**：GPS 接收器输出的标准格式叫 NMEA-0183，最常见的是 `$GPGGA` 语句，记录定位信息。例如：

```
$GPGGA,123519,4807.038,N,01131.000,E,1,08,0.9,545.4,M,46.9,M,,*47
```

各字段含义：
- `$GPGGA`：语句类型
- `123519`：UTC 时间 12:35:19
- `4807.038`：纬度（4807.038 表示 48 度 07.038 分）
- `N`：北纬
- `01131.000`：经度（01131.000 表示 011 度 31.000 分）
- `E`：东经
- 后面是定位质量、卫星数、海拔等

**练习题**：编写代码解析以下 GPS 语句，输出格式化的定位信息：

```python
nmea = "$GPGGA,123519,4807.038,N,01131.000,E,1,08,0.9,545.4,M"
```

**任务**：
1. 用 `split(",")` 把语句拆分成各字段；
2. 取出时间字段（第1位），格式化输出为 `12:35:19`（提示：用切片取每两位）；
3. 取出纬度字段，转换成十进制度（提示：前两位是度，后面是分；公式：度 + 分/60）；
4. 取出经度字段，转换成十进制度（提示：前三位是度）；
5. 取出海拔（第9位）；
6. 用 f-string 输出一份完整的定位报告：

```
==============================
GPS 定位信息
==============================
时间：12:35:19 UTC
纬度：48.1173° N
经度：11.5167° E
海拔：545.4 米
==============================
```

**这是本节课最具挑战的练习**，请认真完成。卡住的同学先看 [4.3 完整版] 那段代码作参考。

In [ ]:
# ===== 即时练习③ =====
# 在下方编写你的代码：

nmea = "$GPGGA,123519,4807.038,N,01131.000,E,1,08,0.9,545.4,M"





---

## 五、本节课小结

### 知识点回顾

| 知识点 | 关键语法 |
|---|---|
| 字符串索引 | `s[n]`、`s[-1]`（负索引从右数） |
| 字符串切片 | `s[a:b]`（左闭右开）、`s[::-1]`（反转） |
| **split** | `s.split(分隔符)` → 列表（不传参 = 按空白分割） |
| **strip** | `s.strip()` 去两端空白；`s.strip("#")` 去指定字符 |
| **replace** | `s.replace(旧, 新)` |
| **join** | `分隔符.join(列表)` |
| **find** | `s.find(子串)` 找位置；找不到返回 -1 |
| input 进阶 | `input("提示语：")` + `float()/int()` 转换 |
| print 进阶 | `print(..., sep=",", end="")` |
| 转义字符 | `\n`(换行)、`\t`(Tab)、`\\`、`\"` |
| 原始字符串 | `r"..."` 不解释转义（路径处理） |

### 重点强调

1. ⭐⭐⭐ **5大方法是地学数据清洗的'瑞士军刀'**——以后处理任何字符串数据都靠它们；
2. ⭐⭐⭐ **切片左闭右开**——这条规律将来 NumPy / Pandas 都遵循；
3. ⭐⭐ **split 不传参数**——自动处理混乱的空白字符，超实用；
4. ⭐⭐ **链式调用**——`s.replace().replace().split()` 把多步操作连成一行；
5. ⭐ **路径用原始字符串 `r"..."`**——避免 Windows 反斜杠的坑。

### 课后作业

**1. 【基础】文件名信息提取（必做）**

给定一组 Landsat 文件名：

```python
files = [
    "LC08_L1TP_123032_20240115_T1.tif",
    "LC09_L1TP_123032_20240220_T1.tif",
    "LC08_L1TP_124033_20240310_T1.tif",
]
```

对每一个文件名（**手动一个个处理**，下次课才学循环），提取并输出：
- 卫星类型（前4位，如 LC08）
- 行列号（如 123032，从字符串中切片提取）
- 拍摄日期（YYYYMMDD 格式，再用切片格式化为 YYYY-MM-DD）

**2. 【进阶】简易日志解析（必做）**

给定一行日志：

```python
log = "[INFO] 2024-03-15 14:30:25 - 数据下载完成: landsat_clip.tif (大小: 45.6MB)"
```

请编写代码提取：
- 日志级别（INFO）
- 日期（2024-03-15）
- 时间（14:30:25）
- 文件名（landsat_clip.tif）
- 文件大小（45.6，转成 float）

并用 f-string 格式化输出。

**3. 【进阶】带方向的度分秒坐标转换（必做）**

扩展课堂的度分秒解析代码，使其能处理带方向标识的坐标，例如：
- `"39°54'27\"N"` → `39.9075`（北纬，正数）
- `"39°54'27\"S"` → `-39.9075`（南纬，负数）
- `"116°24'27\"E"` → `116.4075`（东经，正数）
- `"116°24'27\"W"` → `-116.4075`（西经，负数）

**提示**：先用切片或 `[-1]` 取出方向字符，再决定结果的正负号。

---

下节课预告：**第4次课 条件判断** —— 学习 if/elif/else，让程序学会'根据情况做不同的事'，开启编程的新世界！

**温馨提醒**：第4次课是**零基础学生第一道大坎**。请务必把今天和前两次课的内容消化好——基础越扎实，越能撑过那道坎。

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**
>
> 字符串处理的精髓在于'多试错、多查文档'，直接看答案学不到东西。

In [ ]:
# ===== 即时练习① 参考答案 =====

s = "NorthLatitude39.9075E116.4074"

# 1. 第一个字符
print(s[0])         # N

# 2. 切片取 "NorthLatitude"（前13个字符）
print(s[0:13])      # NorthLatitude
# 也可以写成 s[:13]

# 3. 切片取 "39.9075"（位置13到20）
print(s[13:20])     # 39.9075

# 4. 切片取 "116.4074"（位置21到29）
print(s[21:29])     # 116.4074

# 5. 最后一个字符
print(s[-1])        # 4

# 6. 反转
print(s[::-1])      # 4704.611E5709.93edutitaLhtroN

In [ ]:
# ===== 即时练习② 参考答案 =====

messy_cities = "  杭州市;;北京市；上海市 ；  广州市  "

# Step 1: 把中文分号替换成英文分号
step1 = messy_cities.replace("；", ";")
print("Step 1:", repr(step1))

# Step 2: 用 ; 分割（注意此时空字符串可能也会被分出来）
step2 = step1.split(";")
print("Step 2:", step2)

# Step 3: 对每个城市名 strip
# 因为还没学循环，这里手动一个一个 strip
city0 = step2[0].strip()
city1 = step2[1].strip()    # 这一项可能是空字符串
city2 = step2[2].strip()
city3 = step2[3].strip()
city4 = step2[4].strip()

# 因为出现了 ;; 会产生一个空字符串元素，我们需要跳过它
# （下次课学了循环和条件判断，就能优雅地处理）
print(f"city0={repr(city0)}")
print(f"city1={repr(city1)}")
print(f"city2={repr(city2)}")
print(f"city3={repr(city3)}")
print(f"city4={repr(city4)}")

# Step 4: 把非空的城市拼接（手动选取非空项）
result = " → ".join([city0, city2, city3, city4])
print("\n最终结果：", result)

# === 更高级的写法（使用列表推导式，下下次课才学）===
# cleaned = [c.strip() for c in step2 if c.strip()]
# result = " → ".join(cleaned)

In [ ]:
# ===== 即时练习③ 参考答案 =====

nmea = "$GPGGA,123519,4807.038,N,01131.000,E,1,08,0.9,545.4,M"

# Step 1: 分割
fields = nmea.split(",")
# fields = ['$GPGGA','123519','4807.038','N','01131.000','E','1','08','0.9','545.4','M']

# Step 2: 时间格式化（123519 → 12:35:19）
time_raw = fields[1]
time_str = f"{time_raw[0:2]}:{time_raw[2:4]}:{time_raw[4:6]}"

# Step 3: 纬度处理（4807.038 → 48 度 + 07.038 分 → 十进制度）
lat_raw = fields[2]            # "4807.038"
lat_dir = fields[3]            # "N"
lat_deg = int(lat_raw[0:2])    # 48
lat_min = float(lat_raw[2:])   # 7.038
lat_decimal = lat_deg + lat_min / 60

# Step 4: 经度处理（01131.000 → 011 度 + 31.000 分 → 十进制度）
lon_raw = fields[4]            # "01131.000"
lon_dir = fields[5]            # "E"
lon_deg = int(lon_raw[0:3])    # 11
lon_min = float(lon_raw[3:])   # 31.0
lon_decimal = lon_deg + lon_min / 60

# Step 5: 海拔
altitude = float(fields[9])

# Step 6: 输出报告
border = "=" * 30
print(border)
print("GPS 定位信息")
print(border)
print(f"时间：{time_str} UTC")
print(f"纬度：{lat_decimal:.4f}° {lat_dir}")
print(f"经度：{lon_decimal:.4f}° {lon_dir}")
print(f"海拔：{altitude} 米")
print(border)

---

## 课后作业参考答案

**仅供参考，建议先自己完成。**

In [ ]:
# ===== 课后作业 1：文件名信息提取 =====

# 三个文件名（手动一个个处理）
f1 = "LC08_L1TP_123032_20240115_T1.tif"
f2 = "LC09_L1TP_123032_20240220_T1.tif"
f3 = "LC08_L1TP_124033_20240310_T1.tif"

# 处理 f1
satellite_1 = f1[0:4]                    # LC08
path_row_1 = f1[10:16]                   # 123032
date_raw_1 = f1[17:25]                   # 20240115
date_fmt_1 = f"{date_raw_1[0:4]}-{date_raw_1[4:6]}-{date_raw_1[6:8]}"

print(f"文件1：{f1}")
print(f"  卫星：{satellite_1}, 行列：{path_row_1}, 日期：{date_fmt_1}")

# 处理 f2
satellite_2 = f2[0:4]
path_row_2 = f2[10:16]
date_raw_2 = f2[17:25]
date_fmt_2 = f"{date_raw_2[0:4]}-{date_raw_2[4:6]}-{date_raw_2[6:8]}"

print(f"\n文件2：{f2}")
print(f"  卫星：{satellite_2}, 行列：{path_row_2}, 日期：{date_fmt_2}")

# 处理 f3
satellite_3 = f3[0:4]
path_row_3 = f3[10:16]
date_raw_3 = f3[17:25]
date_fmt_3 = f"{date_raw_3[0:4]}-{date_raw_3[4:6]}-{date_raw_3[6:8]}"

print(f"\n文件3：{f3}")
print(f"  卫星：{satellite_3}, 行列：{path_row_3}, 日期：{date_fmt_3}")

# === 思考题：用 split('_') 也能更优雅地解析（提示）===
# parts = f1.split('_')
# satellite = parts[0]   # LC08
# path_row = parts[2]    # 123032
# date_raw = parts[3]    # 20240115

In [ ]:
# ===== 课后作业 2：日志解析 =====

log = "[INFO] 2024-03-15 14:30:25 - 数据下载完成: landsat_clip.tif (大小: 45.6MB)"

# 思路：综合运用 split / find / replace / 切片

# 提取日志级别 [INFO] -> INFO
level = log[1:5]    # 切片，从位置1取4个字符
# 也可以：level = log.split("]")[0].replace("[", "")

# 提取日期：从位置 7 开始连续 10 个字符
date = log[7:17]    # 2024-03-15

# 提取时间：从位置 18 开始连续 8 个字符
time = log[18:26]   # 14:30:25

# 提取文件名：在 ': ' 和 ' (' 之间
# 思路：先找到 ': ' 的位置，再找到 ' (' 的位置，切片取中间
start = log.find(": ") + 2     # 跳过 ': '
end = log.find(" (")
filename = log[start:end]      # landsat_clip.tif

# 提取大小：在 '大小: ' 之后到 'MB' 之前
size_start = log.find("大小: ") + 4   # 跳过 "大小: "（注意冒号后有空格）
size_end = log.find("MB")
size_str = log[size_start:size_end]
size = float(size_str)

# 输出
print("=== 日志解析结果 ===")
print(f"级别：{level}")
print(f"日期：{date}")
print(f"时间：{time}")
print(f"文件：{filename}")
print(f"大小：{size} MB")

In [ ]:
# ===== 课后作业 3：带方向的度分秒坐标转换 =====

# 测试用例
coord1 = "39°54'27\"N"
coord2 = "39°54'27\"S"
coord3 = "116°24'27\"E"
coord4 = "116°24'27\"W"

# 处理 coord1
direction = coord1[-1]                              # 取最后一个字符（方向）
main_part = coord1[:-1]                             # 去掉方向
parts = main_part.replace("°", " ").replace("'", " ").replace("\"", " ").split()
deg = int(parts[0])
minute = int(parts[1])
second = int(parts[2])
decimal = deg + minute / 60 + second / 3600

# 根据方向决定正负号
if direction == "S" or direction == "W":
    decimal = -decimal

print(f"{coord1} → {decimal:.4f}°")

# 注意：if 还是下次课内容，但你应该能直觉理解上面这两行代码的含义

**注意**：作业3用到了 `if` 条件判断——这是下节课的内容。这里给出来是为了让你**先有个直觉**：

```python
if direction == "S" or direction == "W":
    decimal = -decimal
```

意思是：**如果方向是 S（南）或 W（西），就把数值变成负数**。

下节课我们就会系统地学习这种'根据情况做不同的事'的代码逻辑。